# Generative AI: Assignment 1
## Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction (55 marks)

We use the Job Title and Job Description dataset (`job_title_des.csv`) and a LangChain-powered pipeline to perform: 1) Job Category Classification, and 2) Key Requirements Extraction (skills, education, experience) for each job posting.

## Step 1: Load the Dataset

In [1]:
import os, json, re, getpass
import pandas as pd
from dotenv import load_dotenv

load_dotenv(r"C:\Dhiren\Jio Institute\Course\Term 4\Gen AI\Codes\GenAI\.env", override=True)

True

In [2]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "not-needed-for-ollama"  # not used by the Ollama backend below

In [3]:
if os.environ["GROQ_API_KEY"]:
    print(f"Groq API Key exists and begins {os.environ['GROQ_API_KEY'][:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


In [4]:
df_jobs = pd.read_csv("job_title_des.csv")
df_jobs = df_jobs.drop(columns=["Unnamed: 0"])
df_jobs.columns = df_jobs.columns.str.strip().str.replace(" ", "_")
print(df_jobs.shape)
df_jobs.head()

(2277, 2)


              Job_Title                                    Job_Description
0     Flutter Developer  We are looking for hire experts flutter develo...
1      Django Developer  PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2      Machine Learning  Data Scientist (Contractor)\r\n\r\nBangalore, ...
3         iOS Developer  JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4  Full Stack Developer  job responsibility full stack engineer – react...

In [5]:
# Limit the data to the first 25 job postings
NUM_JOBS = 25
df_jobs_subset = df_jobs.head(NUM_JOBS).reset_index(drop=True)
df_jobs_subset.head()

              Job_Title                                    Job_Description
0     Flutter Developer  We are looking for hire experts flutter develo...
1      Django Developer  PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2      Machine Learning  Data Scientist (Contractor)\r\n\r\nBangalore, ...
3         iOS Developer  JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4  Full Stack Developer  job responsibility full stack engineer – react...

## Step 2: Define the Job Category Classification Task (10 marks)

In [6]:
#Using LangChain
from langchain.chat_models import init_chat_model

# Note: running via Ollama (local) instead of Groq for this run, since the Groq
# account's on_demand daily token quota (200,000 tokens/day) was exhausted.
# Swap model_name/model_provider back to ("openai/gpt-oss-120b", "groq") once quota resets.
model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0.0)

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

category_template = """Given the following job title and description, categorize the job into one of the following domains: Technology/IT, Finance, Marketing, Healthcare, Education, Sales, Human Resources, Operations, Other.

Here are a couple of examples to guide you:
Job: Senior Data Analyst. Description: Looking for a data analyst skilled in SQL and Python to build dashboards and report insights.
Domain category: Technology/IT

Job: Registered Nurse. Description: Provide patient care and support in a busy hospital ward.
Domain category: Healthcare

You may adjust or expand the category if none of the above fit well; use "Other" as a fallback.
Only return the single domain category, nothing else.

Job: {job_title}
Description: {job_description}

Domain category:"""

category_prompt = ChatPromptTemplate.from_template(category_template)
category_chain = category_prompt | llm | StrOutputParser()

In [8]:
#Show this works for a sample datapoint
sample_job = df_jobs_subset.iloc[0]
sample_category = category_chain.invoke({
    "job_title": sample_job["Job_Title"],
    "job_description": sample_job["Job_Description"]
})
print("Job Title:", sample_job["Job_Title"])
print("Predicted category:", sample_category.strip())

Job Title: Flutter Developer
Predicted category: Technology/IT


## Step 3: Define the Requirements Extraction Task (30 marks)

In [9]:
from pydantic import BaseModel, Field

class JobRequirements(BaseModel):
    """Extract the required skills, education level, and years of experience from a job description."""
    skills: list[str] = Field(description="Key skills, programming languages, tools, or domain knowledge required, empty list if not mentioned")
    education: str = Field(description="Minimum education level required or preferred, e.g. Bachelor's degree, MBA; 'Not specified' if not mentioned")
    experience: str = Field(description="Minimum years of experience or experience level required, e.g. '3+ years'; 'Not specified' if not mentioned")

In [10]:
requirements_extractor = llm.with_structured_output(JobRequirements)

In [11]:
#Show this works for a sample datapoint
sample_requirements = requirements_extractor.invoke(f"Job Description:\n{sample_job['Job_Description']}")
sample_requirements

JobRequirements(skills=['Flutter'], education='Not specified', experience='1 year (Preferred)')

## Step 4: Apply the LLM Chain to Each Job Posting (10 marks)

## Step 5: Update the DataFrame with New Columns (5 marks)

We combine the category classification and requirements extraction into a single structured-output schema so only one LLM call is needed per posting, then loop over the DataFrame.

In [12]:
class JobPostingAnalysis(BaseModel):
    """Classify a job posting into a domain category and extract its required skills, education, and experience."""
    category: str = Field(description="Domain category: one of Technology/IT, Finance, Marketing, Healthcare, Education, Sales, Human Resources, Operations, Other")
    skills: list[str] = Field(description="Key required skills/technologies, empty list if not mentioned")
    education: str = Field(description="Minimum education level required/preferred, 'Not specified' if not mentioned")
    experience: str = Field(description="Minimum years of experience or experience level required, 'Not specified' if not mentioned")

posting_extractor = llm.with_structured_output(JobPostingAnalysis)

In [13]:
results = []

for index, row in df_jobs_subset.iterrows():
    try:
        job_prompt = f"Job Title: {row['Job_Title']}\n\nJob Description:\n{row['Job_Description']}"
        analysis = posting_extractor.invoke(job_prompt)
        results.append(analysis.model_dump())
    except Exception as e:
        print(f"Failed at row {index}: {e}")
        results.append({"category": None, "skills": None, "education": None, "experience": None})

Failed at row 2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmq405y6ejzrpfg7qa3q7z39` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7670, Requested 889. Please try again in 4.1925s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Failed at row 8: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmq405y6ejzrpfg7qa3q7z39` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7493, Requested 973. Please try again in 3.495s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Failed at row 9: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmq405y6ejzrpfg7qa3q7z39` servi

In [14]:
results_df = pd.DataFrame(results)
results_df.rename(columns={
    "category": "Predicted_Category",
    "skills": "Required_Skills",
    "education": "Education_Required",
    "experience": "Experience_Required"
}, inplace=True)
results_df.head()

  Predicted_Category  ...                                Experience_Required
0      Technology/IT  ...                                 1 year (Preferred)
1      Technology/IT  ...                                      Not specified
2               None  ...                                               None
3      Technology/IT  ...                                      Not specified
4      Technology/IT  ...  5+ years web development experience, including...

[5 rows x 4 columns]

In [15]:
#Final Pandas dataframe with all original and new columns together
df_final_part2 = pd.concat([df_jobs_subset.reset_index(drop=True), results_df], axis=1)
df_final_part2

                 Job_Title  ...                                Experience_Required
0        Flutter Developer  ...                                 1 year (Preferred)
1         Django Developer  ...                                      Not specified
2         Machine Learning  ...                                               None
3            iOS Developer  ...                                      Not specified
4     Full Stack Developer  ...  5+ years web development experience, including...
5           Java Developer  ...                    2 years of software development
6     Full Stack Developer  ...  Minimum 2 years of full stack development expe...
7     JavaScript Developer  ...                                          3-8 years
8          DevOps Engineer  ...                                               None
9        Software Engineer  ...                                               None
10  Database Administrator  ...  Minimum 9 years of related database administra...
11  

In [16]:
# Spot-check a few rows for correctness
df_final_part2[["Job_Title", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]].sample(5, random_state=42)

                 Job_Title  ...                                Experience_Required
8          DevOps Engineer  ...                                               None
16     Wordpress Developer  ...                                               None
0        Flutter Developer  ...                                 1 year (Preferred)
23  Database Administrator  ...  2 years of database design/administration expe...
11        Machine Learning  ...                                          2-4 years

[5 rows x 5 columns]